# Data-poisoning a **neural-network** victim

The victim is a small MLP trained by online mini-batch SGD on the squared loss

$$f(x;\theta) = W_2\,\tanh(W_1x + b_1) + b_2,\qquad \ell_i = \tfrac12\big(f(x_i;\theta)-y_i\big)^2$$

where $\theta$ is the **flattened parameter vector**. With `d=5` and `victim_hidden=4`
that is $D = 4\cdot5 + 4 + 4 + 1 = 29$ parameters.

**Action-linearity still holds — and this is the whole point.** One SGD step on the
reweighted batch loss is

$$\theta_{t+1} = \theta_t - \tfrac{\eta}{m}\sum_i a_i\nabla_\theta\ell_i(\theta_t)
= \theta_t + \sum_i a_i\varphi_i$$

for **any** per-point loss, however nonlinear the model. The batch gradient is a *sum over
points*, so deleting/duplicating a point adds or removes its term exactly. The
decomposition never needed the victim to be linear — only its *update rule* to be.
(§1 verifies this numerically.)

What does change:

| | linear / logistic | MLP |
|---|---|---|
| `θ` lives in | data space, $\mathbb{R}^d$ | **parameter space**, $\mathbb{R}^{29}$ |
| `phi_batch` | closed form `resid·x` | **per-sample autograd** (`torch.func.vmap(grad)`) |
| start `θ₀` | zeros | small random init (zeros are a degenerate fixed point) |
| stability | robust | SGD **diverges** for `eta ≳ 1` — see §5 |

In [ ]:
import time
from dataclasses import replace

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from config import EnvConfig, MethodConfig, TIER_NAME
from env import Environment, is_feasible
from fvi import train_fvi, evaluate, rollout
from agents import NullAgent, RandomAgent, GreedyAgent, MethodAgent
from decomposition import scores_and_interaction

# NB: do not `import viz` / `import diagnostics` -- they call matplotlib.use("Agg")
# at import time, which disables inline plots.
plt.rcParams["figure.dpi"] = 110
print("torch", torch.__version__)

## Configuration

`eta=0.3` sits inside the stable window (§5); above `eta≈0.5` the victim's own SGD
diverges and cost explodes. `victim_hidden` is deliberately small — `D` is the value
network's input dimension.

Runtime ≈ 1–2 min for the main experiment (per-sample gradients make `phi_batch` costlier
than the closed-form tasks).

In [ ]:
ECFG = EnvConfig(d=5, m=10, T=10, eta=0.3, noise=0.1,
                 task="mlp", victim_hidden=4, init_scale=0.3)
CFG  = MethodConfig(select_mode="sort", c_edit=0.02, weight_decay=1e-3,
                    K_iters=20, rank_r=2)
EVAL_EPISODES, EVAL_SEED = 200, 999

INCLUDE_MODEL_FREE = True   # train BDQ / TRPO too (adds several minutes)
TRPO_ITERS = 50             # TRPO converges later than the others -- see note

torch.manual_seed(0)
env = Environment(ECFG, CFG)
print(f"task={ECFG.task}  data dim d={ECFG.d}  PARAMETER dim D={env.d}  "
      f"hidden={ECFG.victim_hidden}")
print(f"m={env.m}  T={env.T}  eta={ECFG.eta}  c_edit={CFG.c_edit}")

In [ ]:
def compare(env, cfg, V, seeds_rng=1, tiers=("sort", "lowrank")):
    """Evaluate every policy on the SAME held-out episodes."""
    agents = {
        "null":   NullAgent(),
        "random": RandomAgent(torch.Generator().manual_seed(2), cfg.balance, cfg.k_max),
        "greedy": GreedyAgent(env.victim, env.theta_target, cfg, mode="sort"),
    }
    for t in tiers:
        agents[TIER_NAME[t]] = MethodAgent(
            V, env.victim, env.theta_target, replace(cfg, select_mode=t),
            torch.Generator().manual_seed(seeds_rng))
    out = {}
    for n, ag in agents.items():
        c = evaluate(env, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
        e, g = behaviour(env, ag)
        out[n] = {"cost": c, "edits": e, "gap": g, "agent": ag}
    return out


def behaviour(env, agent, n_episodes=60):
    """Mean edits/step and mean ||theta_t - theta_target||; also asserts feasibility."""
    rng = torch.Generator().manual_seed(EVAL_SEED)
    edits = gap = 0.0
    steps = 0
    for _ in range(n_episodes):
        theta = env.theta_0.clone()
        for t in range(env.T):
            z = env.stream.sample_batch(env.m, rng)
            a = agent.act(theta, z, t)
            assert is_feasible(a, CFG.balance, CFG.k_max), "infeasible action!"
            edits += float(np.abs(a.numpy() - 1).sum())
            theta = env.victim.update(theta, z, a).detach()
            gap += float((theta - env.theta_target).norm())
            steps += 1
    return edits / steps, gap / steps


def bar_report(res, title):
    names = list(res)
    colors = {"null": "0.5", "random": "C3", "greedy": "C1",
              "TRPO": "C6", "BDQ": "C2", "first-order": "C0", "low-rank": "C4"}
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
    a1.bar(range(len(names)), [res[n]["cost"] for n in names],
           color=[colors.get(n, "C7") for n in names])
    a1.axhline(res["null"]["cost"], color="0.5", ls="--", lw=1)
    a1.set_xticks(range(len(names))); a1.set_xticklabels(names, rotation=15, fontsize=8)
    a1.set_ylabel("mean episode cost"); a1.set_title(title)
    a2.bar(range(len(names)), [res[n]["edits"] for n in names],
           color=[colors.get(n, "C7") for n in names])
    a2.set_xticks(range(len(names))); a2.set_xticklabels(names, rotation=15, fontsize=8)
    a2.set_ylabel("points edited per step"); a2.set_title("Editing aggressiveness")
    plt.tight_layout(); plt.show()
    nul = res["null"]["cost"]
    for n in names:
        print(f"  {n:18} cost={res[n]['cost']:9.3f}  ({res[n]['cost']/nul:5.3f} x null)"
              f"  edits={res[n]['edits']:5.2f}  gap={res[n]['gap']:7.3f}")

## 1. Verify action-linearity on the MLP

The claim the whole method rests on. We compare `theta + a @ Phi` against an explicit SGD
step on the `a`-reweighted batch loss, computed by autograd through the network.

In [ ]:
rng = torch.Generator().manual_seed(1)
z = env.stream.sample_batch(env.m, rng)
th = env.theta_0.clone()
a = torch.tensor([0., 1., 2., 1., 0., 2., 1., 1., 2., 0.])

lin = th + a @ env.victim.phi_batch(th, z)              # action-linear form
thg = th.detach().clone().requires_grad_(True)
L = (a * 0.5 * (env.victim.forward(thg, z[0]) - z[1]) ** 2).sum() / env.m
grad, = torch.autograd.grad(L, thg)
direct = th - ECFG.eta * grad                            # explicit SGD step

print(f"max |action-linear update  -  explicit SGD step| = "
      f"{float((lin - direct).abs().max()):.3e}   (float32 eps ≈ 1e-7)")
assert float((lin - direct).abs().max()) < 1e-5, "action-linearity broken!"

## 2. Train and compare

In [ ]:
t0 = time.perf_counter()
V, hist = train_fvi(env, CFG, seed=0, verbose=False, return_history=True,
                    best_checkpoint=True, eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
print(f"trained in {time.perf_counter()-t0:.0f}s")

# --- model-free RL baselines, trained on the SAME env / eval episodes ---
if INCLUDE_MODEL_FREE:
    from bdq import train_bdq, BDQAgent
    from trpo import train_trpo, TRPOAgent

    res_mf = {}
    for name, tr, Ag, K in [
                            ("TRPO", train_trpo, TRPOAgent, TRPO_ITERS),
                            ("BDQ",  train_bdq,  BDQAgent,  CFG.K_iters)]:
        cfg_k = replace(CFG, K_iters=K)
        torch.manual_seed(0)
        t0 = time.perf_counter()
        net = tr(env, cfg_k, seed=0, verbose=False, best_checkpoint=True,
                 eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
        ag = Ag(net, env, cfg_k, torch.Generator().manual_seed(1))
        c = evaluate(env, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
        e, g = behaviour(env, ag)
        res_mf[name] = {"cost": c, "edits": e, "gap": g, "agent": ag}
        print(f"  {name:5} (K={K:2d}) cost={c:9.3f}  edits={e:5.2f}  "
              f"[{time.perf_counter()-t0:.0f}s]", flush=True)
else:
    res_mf = {}

res = compare(env, CFG, V)
res.update(res_mf)
ORDER = ["null", "random", "greedy", "TRPO", "BDQ",
         "first-order", "low-rank"]
res = {k: res[k] for k in ORDER if k in res}
bar_report(res, f"MLP victim (D={env.d} params, m={env.m}, T={env.T})")

plt.figure(figsize=(7.5, 4))
plt.plot(range(1, len(hist) + 1), hist, "o-", color="C0", lw=2, label=f"{TIER_NAME[CFG.select_mode]} (FVI)")
plt.axhline(res["null"]["cost"], color="0.5", ls="--", lw=2, label="null")
plt.axhline(res["greedy"]["cost"], color="C1", ls="-.", lw=2, label="greedy")
plt.xlabel("FVI iteration"); plt.ylabel("held-out mean episode cost")
plt.title("Learning curve"); plt.legend(fontsize=9); plt.show()

## 3. The learned function under each attack policy (d = 1)

In 1-D the victim's entire hypothesis is a curve, so the attack is directly visible. A
separate `d=1` run (D = 13 parameters) with **every policy on the identical data stream**.

**Reading the panels.** Marker shape is the attacker's action (● kept, ✕ deleted,
▲ duplicated). Four curves appear in every panel:

| curve | meaning |
|---|---|
| grey dotted | the teacher `y = θ*x` that generated the data |
| **green dashed** | `f(·; θ̃)` — the function the attacker's *target weights* encode |
| blue thin | clean SGD (null) — where the victim lands with no attack |
| **black thick** | `f(·; θ_T)` — where *this* policy drove the victim |

A successful attack pulls the black curve off the blue one and toward the green one. Two
numbers per panel: `‖θ_T − θ̃‖` is the quantity the cost actually minimises, while
**RMSE to the target function** measures the same thing in *function* space — worth
reporting separately, because weight-space symmetry means two different `θ` can encode
nearly the same `f`.

In [ ]:
E1, C1 = replace(ECFG, d=1), replace(CFG, K_iters=20)
torch.manual_seed(0)
env1 = Environment(E1, C1)
print(f"d=1 MLP victim: D = {env1.d} parameters")

V1 = train_fvi(env1, C1, seed=0, verbose=False, best_checkpoint=True,
               eval_episodes=100, eval_seed=EVAL_SEED)

policies = {
    "null (clean SGD)": NullAgent(),
    "random": RandomAgent(torch.Generator().manual_seed(2), C1.balance, C1.k_max),
    "greedy": GreedyAgent(env1.victim, env1.theta_target, C1, mode="sort"),
}
if INCLUDE_MODEL_FREE:            # BDQ dominates the runtime here (per-sample autograd)
    from bdq import train_bdq, BDQAgent
    from trpo import train_trpo, TRPOAgent
    torch.manual_seed(0)
    P1 = train_trpo(env1, replace(C1, K_iters=TRPO_ITERS), seed=0, verbose=False,
                    best_checkpoint=True, eval_episodes=100, eval_seed=EVAL_SEED)
    torch.manual_seed(0)
    Q1 = train_bdq(env1, C1, seed=0, verbose=False, best_checkpoint=True,
                   eval_episodes=100, eval_seed=EVAL_SEED)
    policies["TRPO"] = TRPOAgent(P1, env1, C1, torch.Generator().manual_seed(1))
    policies["BDQ"] = BDQAgent(Q1, env1, C1, torch.Generator().manual_seed(1))
policies[TIER_NAME[C1.select_mode]] = MethodAgent(V1, env1.victim, env1.theta_target, C1,
                                        torch.Generator().manual_seed(1))

rng = torch.Generator().manual_seed(5)
batches = [env1.stream.sample_batch(env1.m, rng) for _ in range(env1.T)]   # shared stream


def run_on(agent):
    th = env1.theta_0.clone()
    xs, ys, acts = [], [], []
    for t, z in enumerate(batches):
        a = agent.act(th, z, t)
        xs.append(z[0].reshape(-1)); ys.append(z[1]); acts.append(a)
        th = env1.victim.update(th, z, a).detach()
    return th, torch.cat(xs).numpy(), torch.cat(ys).numpy(), torch.cat(acts).numpy()


X_all = torch.cat([z[0].reshape(-1) for z in batches]).numpy()
xs_grid = torch.linspace(float(X_all.min()), float(X_all.max()), 300)[:, None]
f = lambda th: env1.victim.forward(th, xs_grid).detach().numpy()
f_star = (xs_grid.reshape(-1) * env1.stream.theta_star).numpy()   # linear teacher
f_targ = f(env1.theta_target)
f_null = f(run_on(NullAgent())[0])

ncols = 3
nrows = (len(policies) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.5 * nrows),
                         sharex=True, sharey=True, squeeze=False)
rows = []
for ax, (name, ag) in zip(axes.flat, policies.items()):
    th, X, Y, A = run_on(ag)
    keep, dele, dup = A == 1, A == 0, A == 2
    ax.scatter(X[keep], Y[keep], s=13, c="0.55", alpha=.45, zorder=2)
    ax.scatter(X[dele], Y[dele], s=75, c="#b3202c", marker="x", linewidths=2.3, zorder=4)
    ax.scatter(X[dup], Y[dup], s=62, c="#1f9c4e", marker="^",
               edgecolors="k", linewidths=.8, zorder=4)
    ax.plot(xs_grid.numpy(), f_star, ":", color="0.35", lw=1.8, zorder=3)
    ax.plot(xs_grid.numpy(), f_targ, "--", color="#00b050", lw=2.4, zorder=5)
    ax.plot(xs_grid.numpy(), f_null, "-", color="tab:blue", lw=1.6, alpha=.75, zorder=5)
    ax.plot(xs_grid.numpy(), f(th), "-", color="k", lw=3.0, zorder=6)
    gap = float((th - env1.theta_target).norm())
    fdist = float(np.sqrt(((f(th) - f_targ) ** 2).mean()))
    rows.append((name, gap, fdist, int(dele.sum()), int(dup.sum())))
    ax.set_title(f"{name}\n‖θ_T − θ̃‖ = {gap:.2f}   RMSE to target fn = {fdist:.2f}\n"
                 f"{int(dele.sum())} deleted / {int(dup.sum())} duplicated", fontsize=9.5)
    ax.set_xlabel("x"); ax.set_ylabel("y / f(x)")
for ax in list(axes.flat)[len(policies):]:
    ax.axis("off")

from matplotlib.lines import Line2D
fig.legend(handles=[
    Line2D([0], [0], color="0.35", ls=":", lw=1.8, label="teacher  y = θ*x"),
    Line2D([0], [0], color="#00b050", ls="--", lw=2.4, label="target  f(·; θ̃)"),
    Line2D([0], [0], color="tab:blue", lw=1.6, label="clean SGD (null)"),
    Line2D([0], [0], color="k", lw=3.0, label="this policy's f(·; θ_T)"),
    Line2D([0], [0], marker="o", color="0.55", ls="None", label="kept"),
    Line2D([0], [0], marker="x", color="#b3202c", ls="None", label="deleted"),
    Line2D([0], [0], marker="^", color="#1f9c4e", mec="k", ls="None", label="duplicated")],
    loc="lower center", ncol=7, fontsize=9)
fig.suptitle(f"MLP victim (d=1, D={env1.d}): the learned function under each attack "
             f"policy — identical data stream", fontsize=13)
fig.tight_layout(rect=(0, 0.07, 1, 0.95)); plt.show()

print(f"{'policy':20}{'||θ_T-θ̃||':>12}{'RMSE to target fn':>19}{'del':>6}{'dup':>6}")
for r in rows:
    print(f"{r[0]:20}{r[1]:>12.3f}{r[2]:>19.3f}{r[3]:>6d}{r[4]:>6d}")
print("\nSingle episode, single seed — illustrative; the quantitative comparison is §2.")

## 4. Interaction diagnostics

Note `rank(B) ≤ min(m, D)`; with `D=29 > m=10` the interaction matrix is generically
**full rank**, unlike the `d < m` linear case where `m-d` eigenvalues vanish.

In [ ]:
alpha_rel, spec = [], []
rng = torch.Generator().manual_seed(EVAL_SEED)
theta = env.theta_0.clone()
for i in range(60):
    t = i % env.T
    if t == 0:
        theta = env.theta_0.clone()
    z = env.stream.sample_batch(env.m, rng)
    alpha, B = scores_and_interaction(V, theta, z, t, env.victim, env.theta_target,
                                      CFG.lam, need_B=True, use_hessian=CFG.use_hessian_in_B)
    a = res["first-order"]["agent"].act(theta, z, t)
    lin = float(a @ alpha)
    alpha_rel.append(abs(float(0.5 * a @ B @ a)) / (abs(lin) + 1e-12))
    spec.append(np.sort(np.abs(np.linalg.eigvalsh(0.5 * (B + B.T).numpy())))[::-1])
    theta = env.victim.update(theta, z, a).detach()

spec = np.array(spec)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
a1.hist(alpha_rel, bins=25, color="C0", alpha=0.85)
a1.axvline(np.median(alpha_rel), color="k", ls="--", lw=1.5,
           label=f"median = {np.median(alpha_rel):.3f}")
a1.set_xlabel("|½ aᵀBa| / |aᵀα|"); a1.set_ylabel("states")
a1.set_title("Relative interaction (what `sort` discards)"); a1.legend(fontsize=9)

a2.semilogy(np.arange(1, spec.shape[1] + 1), spec.mean(0), "o-", color="C0", lw=2)
a2.set_xlabel("index k"); a2.set_ylabel("mean |eigenvalue| of B")
a2.set_title(f"Spectrum of B  (rank(B) ≤ min(m,d) = {min(ECFG.m, env.d)})")
plt.tight_layout(); plt.show()
print(f"relative interaction: median {np.median(alpha_rel):.4f}  mean {np.mean(alpha_rel):.4f}")

## 5. Stability: the victim's own SGD sets the usable `eta`

A nonlinear victim can diverge on its own. Past `eta ≈ 0.5` the MLP's training blows up and
every cost — including `null` — explodes, so nothing about the attack is measurable there.
This is a constraint the linear victim never imposed.

Also watch `greedy` at `eta=0.5`: it becomes **worse than doing nothing**. Myopic editing
destabilises a nonlinear victim, while the planning method still helps.

In [ ]:
print(f"{'eta':>5}{'null':>12}{'greedy':>12}{'first-order':>12}{'g/null':>8}{'m/null':>8}")
etas, ratios_m, ratios_g = [], [], []
for eta in [0.1, 0.2, 0.3, 0.5]:
    e = replace(ECFG, eta=eta)
    torch.manual_seed(0)
    en = Environment(e, CFG)
    Vs = train_fvi(en, CFG, seed=0, verbose=False, best_checkpoint=True,
                   eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
    ag = MethodAgent(Vs, en.victim, en.theta_target, CFG, torch.Generator().manual_seed(1))
    gr = GreedyAgent(en.victim, en.theta_target, CFG, mode="sort")
    n_ = evaluate(en, NullAgent(), torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    g_ = evaluate(en, gr, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    m_ = evaluate(en, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    etas.append(eta); ratios_m.append(m_ / n_); ratios_g.append(g_ / n_)
    print(f"{eta:>5}{n_:>12.3f}{g_:>12.3f}{m_:>12.3f}{g_/n_:>8.3f}{m_/n_:>8.3f}")

plt.figure(figsize=(7.5, 4.2))
plt.plot(etas, ratios_m, "o-", color="C0", lw=2, label=f"{TIER_NAME[CFG.select_mode]} / null")
plt.plot(etas, ratios_g, "s-.", color="C1", lw=2, label="greedy / null")
plt.axhline(1.0, color="0.7", ls=":", lw=1.5, label="no better than null")
plt.xlabel("victim learning rate eta"); plt.ylabel("cost / null (lower = stronger)")
plt.title("Attack strength vs victim stability"); plt.legend(fontsize=9); plt.show()

### Takeaways

1. **Action-linearity is a property of the *update rule*, not the model** (§1 verifies it to
   ~1e-8 on the MLP). The decomposition therefore applies to an arbitrary differentiable
   victim; only `phi_batch` gets more expensive, needing per-sample gradients.
2. **`θ` moves to parameter space.** `env.d` becomes the parameter count `D=29`, and since
   `D > m`, `B` is generically full rank — the low-rank structure that `d < m` gave us in
   the linear case is gone, which is exactly where `lowrank` should start to matter.
3. **The victim's own stability now binds.** Above `eta≈0.5` the MLP diverges and no attack
   is measurable — a constraint linear victims never impose.
4. **Myopia becomes actively harmful.** At `eta=0.5` greedy is *worse than null* (it
   destabilises the victim) while the planning method still reduces cost. This is the
   cleanest evidence yet that cost-to-go matters, and it only appears on a nonlinear victim.

**Caveats.** `θ̃` is a random point in weight space; because of weight-space symmetries
(permuting hidden units leaves `f` unchanged) it is a well-defined but somewhat arbitrary
target — an equally valid formulation would target the victim's *function* rather than its
*weights*. Results are single-seed; re-run with several before citing numbers.